# 📊 DAY 3 — Exploratory Data Analysis (EDA)
**Bluestock Fintech | Capstone Project I — Mutual Fund Analytics**

---

## 🎯 Objectives for Day 3:
1. Visualize daily NAV historical trends across schemes.
2. Analyze AMC dominance and quarterly AUM growth trends.
3. Investigate industry SIP inflows and category allocations.
4. Profile investor demographics (age, gender, income) and geography.
5. Calculate returns correlation and sector concentration risks.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import sqlite3
import os
import warnings
warnings.filterwarnings('ignore')

# Set style configuration
sns.set_theme(style="whitegrid")
plt.rcParams["figure.figsize"] = (12, 6)
plt.rcParams["font.family"] = "sans-serif"

print("✅ Libraries imported successfully.")

## 📂 Step 1 — Load SQLite Database Connection

In [ ]:
BASE_DIR = os.path.abspath(os.path.join(os.getcwd(), '..'))
DB_PATH = os.path.join(BASE_DIR, 'data', 'bluestock_mf.db')
CHARTS_DIR = os.path.join(BASE_DIR, 'outputs', 'charts')
os.makedirs(CHARTS_DIR, exist_ok=True)

conn = sqlite3.connect(DB_PATH)
print(f"🔗 Connected to SQLite Database at: {DB_PATH}")

## 📈 Step 2 — NAV Trend Analysis (2022 - 2026)

Visualizing historical daily NAV prices of select schemes to identify macroeconomic growth runs.

In [ ]:
query = """
SELECT date, amfi_code, nav 
FROM fact_nav 
WHERE amfi_code IN (119551, 120503, 118632, 119092, 120841)
"""
df_nav = pd.read_sql(query, conn)
df_nav['date'] = pd.to_datetime(df_nav['date'])
df_pivot = df_nav.pivot(index='date', columns='amfi_code', values='nav')

plt.figure(figsize=(14, 7))
for col in df_pivot.columns:
    plt.plot(df_pivot.index, df_pivot[col], label=f"Scheme AMFI {col}", alpha=0.85)
plt.title("Historical NAV Trends (Top Schemes) — 2022-2026", fontsize=14, fontweight='bold')
plt.xlabel("Timeline")
plt.ylabel("Net Asset Value (NAV) in INR")
plt.legend()
plt.tight_layout()
plt.savefig(os.path.join(CHARTS_DIR, "nav_trends.png"))
plt.show()

## 🏢 Step 3 — Quarterly AUM Analysis by Fund House

Investigating assets under management market share across various AMCs.

In [ ]:
query = """
SELECT fund_house, SUM(aum_crore) as total_aum 
FROM fact_performance 
GROUP BY fund_house 
ORDER BY total_aum DESC
"""
df_aum = pd.read_sql(query, conn)

plt.figure(figsize=(12, 6))
sns.barplot(data=df_aum, x='total_aum', y='fund_house', palette='viridis')
plt.title("Total Assets Under Management (AUM) by Fund House", fontsize=14, fontweight='bold')
plt.xlabel("Total AUM (in Crores)")
plt.ylabel("Fund House / AMC")
plt.tight_layout()
plt.savefig(os.path.join(CHARTS_DIR, "aum_by_fund_house.png"))
plt.show()

## 👥 Step 4 — Investor Demographics Profile

Visualizing geographic distribution and age profiling of the user transactions dataset.

In [ ]:
query = "SELECT state, SUM(amount_inr) as total_invested FROM fact_transactions GROUP BY state ORDER BY total_invested DESC LIMIT 10"
df_geo = pd.read_sql(query, conn)

plt.figure(figsize=(12, 5))
sns.barplot(data=df_geo, x='total_invested', y='state', palette='Blues_r')
plt.title("Top 10 States by Aggregate Investment Capital", fontsize=14, fontweight='bold')
plt.xlabel("Investment Size (in INR)")
plt.ylabel("State")
plt.tight_layout()
plt.savefig(os.path.join(CHARTS_DIR, "top_states_investment.png"))
plt.show()

In [ ]:
query = "SELECT age_group, transaction_type, AVG(amount_inr) as avg_amount FROM fact_transactions GROUP BY age_group, transaction_type"
df_age = pd.read_sql(query, conn)

plt.figure(figsize=(12, 6))
sns.barplot(data=df_age, x='age_group', y='avg_amount', hue='transaction_type', palette='muted')
plt.title("Average Transaction size by Age Bracket & Type", fontsize=14, fontweight='bold')
plt.xlabel("Age Group Groupings")
plt.ylabel("Average Amount (INR)")
plt.tight_layout()
plt.savefig(os.path.join(CHARTS_DIR, "age_group_vs_transaction_type.png"))
plt.show()

## 🎯 Step 5 — Portfolio Holding Allocations (Sector-wise)

Evaluating equity concentration risk metrics across industries.

In [ ]:
query = "SELECT sector, COUNT(DISTINCT amfi_code) as scheme_counts FROM fact_portfolio GROUP BY sector ORDER BY scheme_counts DESC LIMIT 8"
df_sector = pd.read_sql(query, conn)

plt.figure(figsize=(8, 8))
plt.pie(df_sector['scheme_counts'], labels=df_sector['sector'], autopct='%1.1f%%', startangle=140, colors=sns.color_palette('pastel'))
plt.title("Sector Diversification Across Registered Schemes", fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig(os.path.join(CHARTS_DIR, "sector_pie_chart.png"))
plt.show()

## 🔄 Step 6 — Closing Connection & Exporting Insights

All figures have been saved to the outputs directory. Closing SQL connections safely.

In [ ]:
conn.close()
print("🔌 SQL Connection safely closed. Day 3 Pipeline successfully completed.")